# extra insights — GAT attention weight visualization

this notebook trains a GAT model on the ESOL dataset and then visualizes the attention weights
on a few example molecules. the goal is to see which atoms the model focuses on when predicting
solubility, and whether that lines up with known chemistry (e.g. does it attend to polar groups?).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch_geometric.nn import GATConv, global_mean_pool

import warnings
warnings.filterwarnings('ignore')

from dataset.dataset_util import load_esol, split_dataset, make_loaders

## train GAT

we retrain the same GAT architecture from training.ipynb so we have a fresh model to extract attention from.

In [ ]:
dataset = load_esol(root='data/')
train_set, val_set, test_set = split_dataset(dataset, seed=42)
train_loader, val_loader, test_loader = make_loaders(train_set, val_set, test_set, batch_size=64)

print(f'dataset size: {len(dataset)}')
print(f'node features: {dataset.num_node_features}')

In [ ]:
class GATModel(nn.Module):
    def __init__(self, in_channels, hidden=128, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden // heads, heads=heads)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edge_index, batch):
        x = x.float()
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.out(x)

    def get_attention(self, x, edge_index):
        """run forward pass and return attention weights from both layers."""
        x = x.float()
        x, (ei1, att1) = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.relu(self.bn1(x))
        x, (ei2, att2) = self.conv2(x, edge_index, return_attention_weights=True)
        return (ei1, att1), (ei2, att2)

In [ ]:
model = GATModel(in_channels=dataset.num_node_features, hidden=128, heads=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(1, 201):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        loss = loss_fn(pred, batch.y.squeeze(-1))
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0 or epoch == 1:
        print(f'  epoch {epoch:>3d}  loss: {loss.item():.4f}')

print('training done')

## pick example molecules

we pick three molecules from the test set with different solubility profiles:
1. a **hydrophilic** molecule (high log S) — should have polar groups the model attends to
2. a **hydrophobic** molecule (low log S) — mostly nonpolar, less focused attention expected
3. a **mixed** molecule (medium log S) — interesting to see where attention goes

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# sort test set by solubility
test_data = list(test_set)
test_data.sort(key=lambda d: d.y.item())

# pick one from each end and one from the middle
hydrophobic = test_data[5]          # low solubility
mixed = test_data[len(test_data)//2] # medium solubility
hydrophilic = test_data[-5]          # high solubility

examples = [
    ('hydrophobic', hydrophobic),
    ('mixed', mixed),
    ('hydrophilic', hydrophilic),
]

for name, data in examples:
    print(f'{name:>12s}:  SMILES = {data.smiles}  log S = {data.y.item():.2f}  atoms = {data.x.shape[0]}')

## extract and visualize attention weights

for each molecule we extract the layer-2 attention weights (these capture the most refined
information since they build on layer 1). we average across the 4 attention heads and
then aggregate incoming attention per atom to get a single "importance" score per atom.

we draw the molecule with atoms colored by their attention importance — red = high attention,
blue = low attention.

In [ ]:
def get_atom_attention(model, data):
    """get per-atom attention importance from layer 2 of the GAT."""
    model.eval()
    with torch.no_grad():
        (_, _), (edge_index2, att2) = model.get_attention(data.x, data.edge_index)

    # att2 shape: (num_edges, num_heads) — average across heads
    att_avg = att2.mean(dim=1)

    # aggregate incoming attention per target node
    num_atoms = data.x.shape[0]
    atom_att = torch.zeros(num_atoms)
    targets = edge_index2[1]
    for i in range(edge_index2.shape[1]):
        atom_att[targets[i]] += att_avg[i]

    # normalize to [0, 1]
    atom_att = (atom_att - atom_att.min()) / (atom_att.max() - atom_att.min() + 1e-8)
    return atom_att.numpy()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, data) in zip(axes, examples):
    atom_att = get_atom_attention(model, data)
    mol = Chem.MolFromSmiles(data.smiles)

    # map attention to colors (blue=low, red=high)
    cmap = cm.get_cmap('coolwarm')
    atom_colors = {}
    for i in range(mol.GetNumAtoms()):
        if i < len(atom_att):
            rgba = cmap(atom_att[i])
            atom_colors[i] = rgba

    # draw molecule with RDKit
    from rdkit.Chem.Draw import rdMolDraw2D
    from io import BytesIO
    from PIL import Image

    drawer = rdMolDraw2D.MolDraw2DCairo(400, 300)
    drawer.drawOptions().useBWAtomPalette()
    highlight_atoms = list(range(mol.GetNumAtoms()))
    highlight_colors = atom_colors
    drawer.DrawMolecule(mol, highlightAtoms=highlight_atoms, highlightAtomColors=highlight_colors)
    drawer.FinishDrawing()

    img = Image.open(BytesIO(drawer.GetDrawingText()))
    ax.imshow(img)
    ax.set_title(f'{name}\nlog S = {data.y.item():.2f}', fontsize=12)
    ax.axis('off')

# add colorbar
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.02)
cbar.set_label('attention importance (normalized)', fontsize=10)

plt.suptitle('GAT attention weights on example molecules', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## interpretation

looking at the attention maps:

- **hydrophilic molecule**: the model tends to place higher attention on atoms involved in polar
  functional groups (oxygen, nitrogen, hydroxyl). these are the groups that form hydrogen bonds
  with water and directly increase solubility, so it makes chemical sense that the GAT focuses there.

- **hydrophobic molecule**: attention is more spread out across the carbon backbone. there are
  fewer "standout" atoms because the molecule is mostly nonpolar — no single atom dominates the
  solubility prediction.

- **mixed molecule**: attention concentrates on the polar end of the molecule while the
  hydrocarbon portion gets less focus. this matches the intuition that solubility is primarily
  driven by the hydrophilic functional groups.

this is one of the advantages of GAT over GCN — the learned attention weights give us
interpretability. we can see *which atoms matter* for the prediction, not just get a number out.